# Deconstructing Domain Shift in Diabetic Retinopathy Grading
## A Dual-Branch Multi-Scale Deep Learning Framework with Ordinal Output Heads & LODO Benchmarking

**Research Objective:** Evaluate domain shift and out-of-domain (OOD) generalization across 6 major Diabetic Retinopathy (DR) benchmarks (**APTOS 2019, DDR, DeepDRiD, IDRiD, Messidor-2, and EyePACS**) for 5-grade ordinal severity classification (`0: No DR`, `1: Mild`, `2: Moderate`, `3: Severe`, `4: Proliferative`).

### Core Architectural & Experimental Innovations:
1. **Dual-Branch Multi-Scale Backbone:**
   - **Global Context Branch:** $300 \times 300$ resolution with ResNet-50 + CBAM (Channel & Spatial Attention) $\to F_{Global}$ (2048-d).
   - **Local MIL Branch:** High-resolution sliding window grid extracting $K=16$ patches of $128 \times 128$ $\to$ EfficientNet-B0 per patch $\to$ 1D Max-Pooling $\to F_{Local}$ (1280-d).
   - **Multimodal Fusion:** Concatenation $[F_{Global}, F_{Local}] \to$ MLP Projection $\to D=512$ latent embedding.
2. **Ablation Studies:**
   - **Preprocessing:** Baseline Denoise vs. Ben Graham Background Subtraction vs. Green Channel + CLAHE.
   - **Ordinal Heads:** Softmax_CE, Softmax_QWK, CORAL, CORN, and Cumulative Link Model with clog-log link (CLM_QWK).
3. **Leave-One-Dataset-Out (LODO) Protocol:**
   - 6-Fold cross-dataset external validation isolating each dataset as an unseen OOD test domain.
   - Strict patient-level grouping (`GroupKFold`) on in-domain training pools to eliminate cross-eye data leakage.

## 1. Environment Initialization & Modular Pipeline Imports

In [ ]:
import os
import sys
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

# Add modular package directory to Python path
current_dir = os.getcwd()
possible_paths = [
    os.path.join(current_dir, "full"),
    os.path.join(current_dir, "pipeline", "full"),
    os.path.join(current_dir, "notebooks", "pipeline", "full"),
    os.path.join(current_dir, "notebooks", "diabetic_retinopathy", "pipeline", "full")
]
for p in possible_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# Import modular pipeline components
from data_loader import (
    get_base_raw_dir, DATASET_CONFIG, load_all_datasets,
    leave_one_dataset_out_generator, get_dr_dataloaders, compute_class_weights
)
from preprocessing import get_torch_transforms
from models import DualBranchMILModel
from losses_and_heads import build_model
from trainer import DRTrainer, calculate_dr_metrics

# Suppress non-critical warnings & set random seeds
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Pipeline Initialized | Hardware Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")

## 2. Load & Inspect Multi-Dataset Distributions

In [ ]:
# Resolve raw data root directory
base_raw_dir = get_base_raw_dir()
print(f"[INFO] Resolved Raw Data Directory: {base_raw_dir}")

# Load all 6 standardized datasets
all_datasets = load_all_datasets(base_dir=base_raw_dir)

# Generate summary audit dataframe
summary_records = []
for name, df in all_datasets.items():
    dist = df["level"].value_counts().to_dict()
    summary_records.append({
        "Dataset": name,
        "Total Images": len(df),
        "Unique Patients": df["patient_id"].nunique(),
        "Grade 0 (No DR)": dist.get(0, 0),
        "Grade 1 (Mild)": dist.get(1, 0),
        "Grade 2 (Moderate)": dist.get(2, 0),
        "Grade 3 (Severe)": dist.get(3, 0),
        "Grade 4 (Proliferative)": dist.get(4, 0)
    })

summary_df = pd.DataFrame(summary_records)
display(summary_df)

## 3. Visual Inspection: Dual-Branch Multi-Scale Sample Batch
Verify the $300 \times 300$ Global Context representation and the $K=16$ local $128 \times 128$ MIL patches extracted from fundus images.

In [ ]:
# Build Preprocessing & Augmentation Pipelines (Mode: Green Channel + CLAHE)
global_train_tf, global_eval_tf, patch_train_tf, patch_eval_tf = get_torch_transforms(
    mode="clahe",
    global_size=300,
    patch_size=128
)

# Create sample DataLoader from first dataset
sample_key = list(all_datasets.keys())[0]
sample_df = all_datasets[sample_key].head(10)

from data_loader import DRMultiScaleDataset
vis_dataset = DRMultiScaleDataset(
    df=sample_df,
    global_transform=global_eval_tf,
    patch_transform=patch_eval_tf,
    num_patches=16,
    patch_size=128,
    global_size=300,
    is_training=False
)

sample_item = vis_dataset[0]
global_img_t = sample_item["global_img"]
patches_t = sample_item["local_patches"]

# Denormalize for matplotlib visualization
mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

g_np = (global_img_t.numpy() * std + mean).clip(0, 1).transpose(1, 2, 0)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(g_np)
axes[0].set_title(f"Global View (300x300)\nGrade: {sample_item['label'].item()}", fontsize=11, fontweight="bold")
axes[0].axis("off")

for p_idx in range(4):
    p_np = (patches_t[p_idx].numpy() * std + mean).clip(0, 1).transpose(1, 2, 0)
    axes[p_idx + 1].imshow(p_np)
    axes[p_idx + 1].set_title(f"MIL Patch #{p_idx+1} (128x128)", fontsize=10)
    axes[p_idx + 1].axis("off")

plt.suptitle(f"{sample_item['dataset']} - Sample Multi-Scale Representation", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 4. Experiment Configuration & Leave-One-Dataset-Out (LODO) Benchmarking

We benchmark our **Champion Model (Dual-Branch ResNet50-CBAM + EfficientNet-B0 MIL + CLM_QWK Head)** across all 6 cross-domain LODO folds to quantify domain shift degradation $\Delta_{QWK} = \text{QWK}_{In-Domain} - \text{QWK}_{OOD}$.

In [ ]:
# Experiment Hyperparameters
CONFIG = {
    "PREPROCESSING_MODE": "clahe",      # 'baseline', 'ben_graham', or 'clahe'
    "HEAD_TYPE": "CLM_QWK",             # 'Softmax_CE', 'Softmax_QWK', 'CORAL', 'CORN', 'CLM_QWK'
    "BATCH_SIZE": 16,
    "NUM_EPOCHS": 15,
    "LEARNING_RATE": 2e-4,
    "WEIGHT_DECAY": 1e-4,
    "NUM_WORKERS": 4,
    "USE_CBAM": True,
    "USE_WEIGHTED_SAMPLER": True,
    "CHECKPOINT_DIR": "./experiment_checkpoints"
}

print("=== EXPERIMENT CONFIGURATION ===")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Initialize LODO Generator
lodo_gen = leave_one_dataset_out_generator(
    dataset_dfs=all_datasets,
    val_ratio=0.15,
    random_state=SEED
)

lodo_results = []

# Execute Leave-One-Dataset-Out (LODO) Cross-Domain Folds
for fold_idx, (train_df, val_df, test_df, ood_key) in enumerate(lodo_gen, 1):
    print(f"\n>>> RUNNING LODO FOLD {fold_idx}/6: OOD Test Set = '{ood_key}' <<<")
    
    # 1. Build Transforms
    g_train_tf, g_eval_tf, p_train_tf, p_eval_tf = get_torch_transforms(
        mode=CONFIG["PREPROCESSING_MODE"],
        global_size=300,
        patch_size=128
    )
    
    # 2. Build DataLoaders
    train_loader, val_loader, test_loader = get_dr_dataloaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        global_train_transform=g_train_tf,
        global_eval_transform=g_eval_tf,
        patch_train_transform=p_train_tf,
        patch_eval_transform=p_eval_tf,
        batch_size=CONFIG["BATCH_SIZE"],
        num_workers=CONFIG["NUM_WORKERS"],
        use_weighted_sampler=CONFIG["USE_WEIGHTED_SAMPLER"]
    )
    
    # 3. Instantiate Dual-Branch Model with Head
    model = build_model(
        head_type=CONFIG["HEAD_TYPE"],
        pretrained=True,
        use_cbam=CONFIG["USE_CBAM"],
        fusion_dim=512,
        num_classes=5,
        qwk_weight=1.2
    )
    
    # 4. Compute Class Weights for Imbalance Loss Adjustment
    class_weights = compute_class_weights(train_df, num_classes=5)
    
    # 5. Optimizer & Cosine Annealing LR Scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["LEARNING_RATE"],
        weight_decay=CONFIG["WEIGHT_DECAY"]
    )
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG["NUM_EPOCHS"],
        eta_min=1e-6
    )
    
    # 6. Train & Evaluate
    trainer = DRTrainer(
        model=model,
        optimizer=optimizer,
        lr_scheduler=lr_scheduler,
        device=DEVICE,
        use_amp=True,
        class_weights=class_weights,
        checkpoint_dir=CONFIG["CHECKPOINT_DIR"]
    )
    
    result = trainer.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=CONFIG["NUM_EPOCHS"],
        model_name=f"lodo_{ood_key}_{CONFIG['HEAD_TYPE']}"
    )
    
    ood_metrics = result["ood_test_metrics"]
    lodo_results.append({
        "OOD_Dataset": ood_key,
        "In_Domain_Val_QWK": result["best_val_qwk"],
        "OOD_Test_QWK": ood_metrics.get("QWK", 0.0),
        "Delta_QWK_Drop": ood_metrics.get("Delta_QWK", 0.0),
        "OOD_Within_1_Acc": ood_metrics.get("Within_1_Grade_Acc", 0.0),
        "OOD_Ref_Sensitivity": ood_metrics.get("Referable_Sensitivity", 0.0),
        "OOD_Ref_Specificity": ood_metrics.get("Referable_Specificity", 0.0),
        "Monotonicity_Violations": ood_metrics.get("Monotonicity_Violations", 0.0)
    })
    
    # Break early after Fold 1 if running in quick prototype testing mode
    # To run all 6 folds, comment out the line below
    # break

print("\n[INFO] LODO Evaluation Completed.")

## 5. Deconstructing Domain Shift: Results & Cross-Domain Performance Audit

In [ ]:
# Convert LODO results into structured benchmark summary table
results_df = pd.DataFrame(lodo_results)
display(results_df)

# Plot Domain Shift Drop (Delta QWK) per Dataset
if not results_df.empty and len(results_df) > 1:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=results_df, x="OOD_Dataset", y="Delta_QWK_Drop", palette="viridis")
    plt.title("Domain Shift Degradation ($\Delta_{QWK} = QWK_{In-Domain} - QWK_{OOD}$)", fontsize=13, fontweight="bold")
    plt.ylabel("QWK Drop (Lower is Better Generalization)", fontsize=11)
    plt.xlabel("Held-Out Out-of-Domain (OOD) Dataset", fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()